# Longevity risk in a life annuity portfolio: stochastic valuation versus the Solvency II standard formula

An insurer pays life annuities-due of 1 per year to policyholders of a given age. We measure longevity risk with
**official data**: Eurostat deaths and population (mortality) and the ECB euro area AAA yield curve (discounting).

1. Lee-Carter model with a random walk with drift (see `notebooks/mortality_projection/lee_carter_mortality_projection.ipynb`).
2. Risk-free curve: ECB Svensson curve up to the last liquid point, extrapolated with the **Smith-Wilson** method used by EIOPA.
3. Best estimate with period and cohort (projected) mortality.
4. Run-off distribution of the annuity value (systematic longevity risk, with and without parameter uncertainty).
5. Idiosyncratic risk and the pooling effect for portfolios of 100 to 10,000 lives.
6. Capital requirement: **one-year 99.5% VaR** with re-estimation of the trend (Richards, Currie and Ritchie, 2014) versus the
   **standard-formula longevity shock** (permanent 20% decrease of mortality rates, Delegated Regulation (EU) 2015/35, Article 138).
7. Sensitivity of the standard-formula requirement to the level of interest rates.

The Monte Carlo simulations (up to $10^8$ simulated lifetimes) run in the C++ engine; NumPy reference implementations are used for checks.

**Conventions.** Valuation at 31 December of the last calibration year $T$; ages are exact at valuation; the annuity-due pays 1 at
$t = 0, 1, \dots$ while the annuitant is alive; discount factors from the Smith-Wilson curve; no expenses, no lapses, no
basis risk between the population and the insured portfolio. The UFR of 3.30% (annual compounding) is the EIOPA value
for the euro applied in 2024; check the current value on the EIOPA website.

**Data modes.** `LONGEVITY_RISK_DATA_MODE=synthetic` runs offline with simulated mortality and an illustrative curve (not official data).

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Repository root (used for data and output paths).
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "longevity_risk").is_dir())
try:
    import longevity_risk
except ImportError:  # not installed: use the source folder (the extension must be built in place)
    sys.path.insert(0, str(ROOT / "scripts" / "longevity_risk"))
    import longevity_risk
from longevity_risk import curves, data, lee_carter as lc, life_table as lt, simulation as sim, solvency, synthetic

core = longevity_risk.require_cpp()

DATA_MODE = os.environ.get("LONGEVITY_RISK_DATA_MODE", "official")  # "official" (Eurostat + ECB) or "synthetic"
COUNTRY, SEX = "IT", "M"             # Eurostat country and sex of the annuitants
AGES = range(50, 100)
FIRST_YEAR = 1975
PANDEMIC_YEARS = (2020, 2021, 2022)  # zero weight in the Lee-Carter fit
UFR, LLP = 0.033, 20                 # ultimate forward rate (annual compounding) and last liquid point, in years
VALUATION_AGES = (55, 65, 75, 85)
N_SCENARIOS = 100_000
SEED = 20240101
OUTPUT_DIR = ROOT / "outputs" / "longevity_scr"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print(f"longevity_risk {longevity_risk.__version__} | data mode: {DATA_MODE} | {COUNTRY} {SEX} | threads: {os.cpu_count()}")

## 1. Mortality model

In [ ]:
if DATA_MODE == "official":
    D, E = data.load_deaths_exposures(COUNTRY, SEX, AGES)
else:
    D, E, _ = synthetic.deaths_and_exposures(AGES, range(1975, 2024), seed=7)
keep = [y for y in D.columns if y >= FIRST_YEAR]
D, E = D.loc[:, keep], E.loc[:, keep]
fit = lc.fit_poisson(D, E, fit_years=[y for y in keep if y not in PANDEMIC_YEARS])
model = sim.ProjectionModel.from_fit(fit, omega=120)
rw = model.rw
T = model.valuation_year
print("Source:", D.attrs.get("source"))
print(f"Calibration {rw.first_year}-{T}: drift {rw.drift:.4f} (s.e. {rw.drift_se:.4f}), sigma {rw.sigma:.4f}; "
      f"valuation date 31 December {T}; table closed at age {model.omega}")

## 2. Risk-free curve with Smith-Wilson extrapolation

Zero-coupon prices $P(u_j)$ are read from the ECB Svensson curve at $u_j = 1, \dots, 20$ years. Smith-Wilson solves
$P(t) = e^{-\omega t} + \sum_j \zeta_j W(t, u_j)$ with $\omega = \ln(1 + \text{UFR})$ and the Wilson kernel
$W(t,u) = e^{-\omega(t+u)}\,[\alpha \min(t,u) - e^{-\alpha \max(t,u)} \sinh(\alpha \min(t,u))]$, so that the curve reproduces
the input prices and its forward rates converge to the UFR. $\alpha$ is the smallest value $\ge 0.05$ for which the forward rate
at 60 years is within 1 basis point of the UFR.

In [ ]:
if DATA_MODE == "official":
    svensson = data.load_ecb_svensson_parameters(f"{T}-12-01", f"{T}-12-31")
else:
    svensson = synthetic.svensson_parameters(f"{T}-12-31")
ecb = curves.SvenssonCurve.from_series(svensson.iloc[-1])
sw = curves.smith_wilson_from_svensson(ecb, UFR, LLP)
print("Source:", svensson.attrs.get("source"), "| curve date:", f"{svensson.index[-1]:%Y-%m-%d}")
print(f"Smith-Wilson alpha = {sw.alpha:.4f}; |forward(60y) - UFR| = {1e4 * sw.convergence_gap():.3f} bp")

t = np.arange(1, 101)
ecb_annual = np.expm1(ecb.zero_rate(t))            # continuous -> annual compounding
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t, 100 * ecb_annual, label="ECB Svensson zero rate (extrapolated beyond 30y)")
ax.plot(t, 100 * sw.zero_rate(t), label="Smith-Wilson zero rate")
ax.plot(t, 100 * sw.forward_rate(t), "--", label="Smith-Wilson 1-year forward rate")
ax.axhline(100 * UFR, color="k", lw=0.8, ls=":", label="UFR")
ax.axvline(LLP, color="gray", lw=0.8)
ax.set(xlabel="maturity (years)", ylabel="% (annual compounding)", title=f"Discount curve at 31 December {T}")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "discount_curve.png", dpi=150)

key = [1, 5, 10, 20, 30, 40, 60, 100]
display(pd.DataFrame({"ECB Svensson (%)": 100 * np.expm1(ecb.zero_rate(key)), "Smith-Wilson (%)": 100 * sw.zero_rate(key),
                      "SW discount factor": sw.discount(key)}, index=pd.Index(key, name="maturity")))
discount = sw.discount(np.arange(0, model.omega - min(AGES) + 2))
discount[0] = 1.0

## 3. Best estimate: period versus cohort mortality

The **period** basis freezes mortality at the rates of year $T$; the **cohort** basis follows the central Lee-Carter projection
$k_{T+h} = k_T + h\,d$. The difference is the cost of future mortality improvements.

In [ ]:
rows = {}
for age in VALUATION_AGES:
    H = model.horizon(age)
    q_period = model.cohort_q(age, np.full(H, rw.k_last))
    q_cohort = model.central_q(age)
    rows[age] = {"annuity (period)": lt.annuity_due(q_period, discount),
                 "annuity (cohort)": lt.annuity_due(q_cohort, discount),
                 "curtate e (period)": lt.curtate_life_expectancy(q_period),
                 "curtate e (cohort)": lt.curtate_life_expectancy(q_cohort)}
best = pd.DataFrame(rows).T
best["improvement cost (%)"] = 100 * (best["annuity (cohort)"] / best["annuity (period)"] - 1)
best.index.name = "age"
display(best)

## 4. Run-off distribution of the annuity value at 65

Each scenario simulates the whole future path of $k_t$ and values the annuity with the resulting cohort survival probabilities
(an infinitely large portfolio, so only systematic risk remains). Parameter uncertainty draws the drift from $N(d, \text{s.e.}(d)^2)$.

In [ ]:
AGE = 65
be65 = model.best_estimate(AGE, discount)
runs = {}
for label, pu in (("process risk only", False), ("with parameter uncertainty", True)):
    t0 = time.perf_counter()
    out = sim.simulate_annuity(model, AGE, discount, N_SCENARIOS, seed=SEED, parameter_uncertainty=pu)
    runs[label] = (out["pv_systematic"], time.perf_counter() - t0)
t0 = time.perf_counter()
ref = sim.simulate_annuity_numpy(model, AGE, discount, N_SCENARIOS, seed=SEED)["pv_systematic"]
t_numpy = time.perf_counter() - t0

table = {}
for label, (pv, seconds) in runs.items():
    table[label] = {"mean": pv.mean(), "std. dev.": pv.std(ddof=1), "MC s.e. of mean": pv.std(ddof=1) / np.sqrt(pv.size),
                    "99.5% quantile": np.quantile(pv, 0.995), "(q99.5 - BE) / BE (%)": 100 * (np.quantile(pv, 0.995) / be65 - 1),
                    "seconds (C++)": seconds}
display(pd.DataFrame(table).T)
print(f"Best estimate (central projection): {be65:.4f}")
print(f"NumPy reference with parameter uncertainty: mean {ref.mean():.4f}, 99.5% quantile {np.quantile(ref, 0.995):.4f}, "
      f"{t_numpy:.2f} s")

fig, ax = plt.subplots(figsize=(8, 4))
for label, (pv, _) in runs.items():
    ax.hist(pv, bins=120, density=True, alpha=0.5, label=label)
ax.axvline(be65, color="k", lw=1, label="best estimate")
ax.set(xlabel="present value of the annuity at 65", ylabel="density", title=f"Run-off distribution ({N_SCENARIOS:,} scenarios)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "runoff_distribution.png", dpi=150)

## 5. Idiosyncratic risk and pooling

For a portfolio of $N$ annuitants we simulate each curtate lifetime by inversion, $P(K \ge t) = {}_tp_x$, and average the present
values. The variance of the value per policy is approximately $\operatorname{Var}_{\text{sys}} + \bar\sigma^2_{\text{idio}}/N$:
idiosyncratic risk diversifies away, systematic longevity risk does not.

In [ ]:
n_scen = 10_000
single_var = lt.annuity_due_variance(model.central_q(AGE), discount)
rows = {}
for n_lives in (100, 1_000, 10_000):
    t0 = time.perf_counter()
    out = sim.simulate_annuity(model, AGE, discount, n_scen, n_lives=n_lives, seed=SEED)
    seconds = time.perf_counter() - t0
    total, systematic = out["pv_portfolio"], out["pv_systematic"]
    rows[n_lives] = {"std. dev. per policy": total.std(ddof=1), "systematic part": systematic.std(ddof=1),
                     "idiosyncratic (theory)": np.sqrt(single_var / n_lives),
                     "(q99.5 - BE) / BE (%)": 100 * (np.quantile(total, 0.995) / be65 - 1),
                     "simulated lifetimes": n_scen * n_lives, "seconds (C++)": seconds}
pool = pd.DataFrame(rows).T
pool.index.name = "lives"
display(pool.astype({"simulated lifetimes": int}))

## 6. Capital requirement: one-year VaR versus the standard formula

- **Standard formula**: $\text{SCR} = \text{BE}(0.8\,q) - \text{BE}(q)$.
- **One-year VaR** (Richards, Currie and Ritchie, 2014): simulate the next year's $k_{T+1}$ (with parameter uncertainty), re-estimate
  the drift with the extra observation, revalue the liability at the end of the year with the updated best-estimate projection and
  discount it back: $\text{SCR} = q_{99.5\%}(X) - \text{BE}$.
- **Run-off VaR**: 99.5% quantile of the full-lifetime distribution of section 4 minus BE (a much longer horizon than Solvency II requires).

All figures are in percent of the best estimate.

In [ ]:
rows = {}
for age in VALUATION_AGES:
    be = model.best_estimate(age, discount)
    sf = solvency.longevity_scr(model.central_q(age), discount)
    one_year = sim.one_year_recalibration(model, age, discount, N_SCENARIOS, seed=SEED)["value"]
    runoff = sim.simulate_annuity(model, age, discount, N_SCENARIOS, seed=SEED)["pv_systematic"]
    rows[age] = {"best estimate": be,
                 "standard formula (%)": 100 * sf["scr_ratio"],
                 "one-year VaR 99.5% (%)": 100 * (np.quantile(one_year, 0.995) / be - 1),
                 "run-off VaR 99.5% (%)": 100 * (np.quantile(runoff, 0.995) / be - 1)}
scr = pd.DataFrame(rows).T
scr.index.name = "age"
display(scr)
scr.to_csv(OUTPUT_DIR / "scr_comparison.csv")

fig, ax = plt.subplots(figsize=(8, 4))
width = 0.27
x = np.arange(len(VALUATION_AGES))
for i, col in enumerate(["standard formula (%)", "one-year VaR 99.5% (%)", "run-off VaR 99.5% (%)"]):
    ax.bar(x + (i - 1) * width, scr[col], width, label=col.replace(" (%)", ""))
ax.set_xticks(x, [str(a) for a in VALUATION_AGES])
ax.set(xlabel="age at valuation", ylabel="% of best estimate", title="Longevity capital requirement")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "scr_comparison.png", dpi=150)

## 7. Interest-rate sensitivity of the standard-formula requirement

The longevity shock acts on cash flows far in the future, so its impact grows as discount rates fall (longer duration).

In [ ]:
rows = {}
H65 = model.horizon(AGE)
for label, v in [("Smith-Wilson curve", discount)] + [(f"flat {r:.0%}", (1 + r) ** -np.arange(H65 + 1)) for r in (0.0, 0.01, 0.02, 0.03, 0.04)]:
    out = solvency.longevity_scr(model.central_q(AGE), v)
    rows[label] = {"best estimate": out["best_estimate"], "standard formula (%)": 100 * out["scr_ratio"]}
display(pd.DataFrame(rows).T)

## 8. Interpretation guide

- The cohort basis is more expensive than the period basis: ignoring future improvements understates the reserve, and the gap
  widens for younger annuitants, whose payments extend further into the future.
- Parameter uncertainty in the drift widens the run-off distribution considerably: over long horizons, uncertainty about the trend
  matters more than year-to-year fluctuations.
- Idiosyncratic risk falls roughly as $1/\sqrt{N}$; for large portfolios the capital need is dominated by systematic (trend) risk.
- The one-year VaR captures only the information revealed in one year (a new observation and a revised trend), so it is typically
  well below the run-off VaR. Comparing it with the 20% shock shows whether the standard formula is prudent for the modelled
  population; differences between populations, ages and interest-rate levels matter.
- Limitations: single-factor Lee-Carter without cohort effects, Gaussian random walk, population rather than insured-lives mortality,
  and deterministic interest rates (no joint interest-rate and longevity scenarios).

Tables and figures are saved in `outputs/longevity_scr/`.

### References

- Börger, M. (2010). Deterministic shock vs. stochastic value-at-risk: an analysis of the Solvency II standard model approach to longevity risk. *Blätter der DGVFM*, 31(2), 225-259.
- Brouhns, N., Denuit, M. and Vermunt, J. K. (2002). A Poisson log-bilinear regression approach to the construction of projected lifetables. *Insurance: Mathematics and Economics*, 31(3), 373-393.
- Commission Delegated Regulation (EU) 2015/35 of 10 October 2014 supplementing Directive 2009/138/EC (Solvency II), Article 138.
- EIOPA. Technical documentation of the methodology to derive EIOPA's risk-free interest rate term structures. https://www.eiopa.europa.eu/tools-and-data/risk-free-interest-rate-term-structures_en
- European Central Bank. Euro area yield curves. https://www.ecb.europa.eu/stats/financial_markets_and_interest_rates/euro_area_yield_curves/html/index.en.html
- Eurostat. Deaths by age and sex (`demo_magec`); Population on 1 January by age and sex (`demo_pjan`).
- Lee, R. D. and Carter, L. R. (1992). Modeling and forecasting U.S. mortality. *Journal of the American Statistical Association*, 87(419), 659-671.
- Richards, S. J., Currie, I. D. and Ritchie, G. P. (2014). A value-at-risk framework for longevity trend risk. *British Actuarial Journal*, 19(1), 116-139.
- Smith, A. and Wilson, T. (2001). Fitting yield curves with long term constraints. Research notes, Bacon and Woodrow.